# Vertical Slice Debug for `91243943_0007.dcm`

This notebook focuses on the status-2 behavior for `91243943_0007.dcm`.

What it shows:
- full frame and detected measurement ROI
- segmented line crops
- dead-space splitter binary mask and slice boxes
- full-line OCR on `gray_x3_lanczos`, `adaptive_threshold`, and `clahe`
- per-slice OCR text from the same vertical-slice retry path

Current working hypothesis: the dead-space splitter is producing a small number of coarse vertical runs, not a true character count, so correct full-line text is still marked as `review_required`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle


def find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "app" / "pipeline" / "echo_ocr_pipeline.py").is_file():
            return p
    raise RuntimeError("Open this notebook from the Master repo or a child directory.")


REPO = find_repo(Path.cwd().resolve())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from app.io.dicom_loader import load_dicom_series
from app.ocr.preprocessing import preprocess_gray_x3_lanczos, preprocess_roi
from app.pipeline.echo_ocr_pipeline import (
    DEFAULT_SEGMENTATION_EXTRA_LEFT_PAD_PX,
    DEFAULT_SEGMENTATION_MODE,
    DEFAULT_TARGET_LINE_HEIGHT_PX,
)
from app.pipeline.layout.echo_ocr_box_detector import TopLeftBlueGrayBoxDetector
from app.pipeline.layout.line_segmenter import LineSegmenter
from app.pipeline.ocr.ocr_engines import TesseractEngine
from app.pipeline.transcription.char_slice_ocr_experimental import per_char_slice_ocr_line
from app.pipeline.transcription.dead_space_char_splitter import split_dead_space_char_slices

DICOM_PATH = REPO / "91243943_0007.dcm"
LABELS_PATH = REPO / "labels" / "labels.json"

if not DICOM_PATH.is_file():
    raise FileNotFoundError(DICOM_PATH)

labels_payload = json.loads(LABELS_PATH.read_text(encoding="utf-8"))
expected_lines = []
for item in labels_payload.get("files", []):
    if item.get("file_name") == DICOM_PATH.name:
        expected_lines = [entry["text"] for entry in item.get("measurements", [])]
        break

print("REPO:", REPO)
print("DICOM:", DICOM_PATH)
print("Expected label lines:")
pprint(expected_lines)


In [ ]:
series = load_dicom_series(DICOM_PATH, load_pixels=True)
frame = series.get_frame(0)

detector = TopLeftBlueGrayBoxDetector()
detection = detector.detect(frame)
if not detection.present or detection.bbox is None:
    raise RuntimeError("Measurement ROI was not detected.")

x, y, bw, bh = detection.bbox
roi = frame[y : y + bh, x : x + bw].copy()

segmenter = LineSegmenter(
    segmentation_mode=DEFAULT_SEGMENTATION_MODE,
    target_line_height_px=DEFAULT_TARGET_LINE_HEIGHT_PX,
    extra_left_pad_px=DEFAULT_SEGMENTATION_EXTRA_LEFT_PAD_PX,
)
segmentation = segmenter.segment(roi, tokens=None)
engine = TesseractEngine(psm=7)


def safe_text(image: np.ndarray) -> str:
    try:
        return engine.extract(image).text.replace("\n", " ").strip()
    except Exception as exc:
        return f"<ocr failed: {exc}>"


def splitter_binary(line_image: np.ndarray) -> np.ndarray:
    if line_image.ndim == 3:
        gray = cv2.cvtColor(line_image[..., :3].astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = line_image.astype(np.uint8)
    return cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2.0,
    )


rows = []
line_debug = []
for line in segmentation.lines:
    lx, ly, lw, lh = line.bbox
    crop = roi[ly : ly + lh, lx : lx + lw].copy()
    split = split_dead_space_char_slices(crop)
    vertical_text, mean_conf, min_conf, per_slice_conf = per_char_slice_ocr_line(
        crop,
        split.slices,
        primary_engine=engine,
        fallback_engine=None,
        preprocessor=preprocess_gray_x3_lanczos,
    ) if split.slices else ("", 0.0, 0.0, ())
    gray_view = preprocess_roi(crop, scale_factor=3, scale_algo="lanczos", contrast_mode="none")
    adaptive_view = preprocess_roi(crop, scale_factor=3, scale_algo="lanczos", contrast_mode="adaptive_threshold")
    clahe_view = preprocess_roi(crop, scale_factor=3, scale_algo="lanczos", contrast_mode="clahe")
    row = {
        "line_order": line.order,
        "bbox": line.bbox,
        "gray_text": safe_text(gray_view),
        "adaptive_text": safe_text(adaptive_view),
        "clahe_text": safe_text(clahe_view),
        "expected_slice_count": split.expected_char_count,
        "gap_count": split.gap_count,
        "split_confidence": round(float(split.confidence), 3),
        "vertical_slice_text": vertical_text,
        "vertical_mean_conf": round(float(mean_conf), 3),
        "vertical_min_conf": round(float(min_conf), 3),
        "slice_boxes": [(s.x, s.y, s.width, s.height) for s in split.slices],
    }
    rows.append(row)
    line_debug.append(
        {
            "row": row,
            "crop": crop,
            "split": split,
            "binary": splitter_binary(crop),
            "gray_view": gray_view,
            "adaptive_view": adaptive_view,
            "clahe_view": clahe_view,
        }
    )

print("Detected ROI bbox:", detection.bbox)
print("Segmented lines:", len(segmentation.lines))
print()
for row in rows:
    pprint(row)
    print("-" * 100)


In [ ]:
frame_overlay = frame.copy()
cv2.rectangle(frame_overlay, (x, y), (x + bw, y + bh), (0, 255, 0), 2)

roi_overlay = roi.copy()
for line in segmentation.lines:
    lx, ly, lw, lh = line.bbox
    cv2.rectangle(roi_overlay, (lx, ly), (lx + lw, ly + lh), (0, 255, 255), 1)
    cv2.putText(
        roi_overlay,
        str(line.order),
        (lx, max(0, ly - 2)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.35,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(frame_overlay)
axes[0].set_title("Full frame with ROI")
axes[1].imshow(roi_overlay)
axes[1].set_title("ROI with segmented lines")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


def show_line_debug(line_order: int) -> None:
    entry = next(item for item in line_debug if item["row"]["line_order"] == line_order)
    row = entry["row"]
    split = entry["split"]
    crop = entry["crop"]
    binary = entry["binary"]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(crop)
    axes[0].set_title(f"Line {line_order} raw crop")

    axes[1].imshow(binary, cmap="gray")
    axes[1].set_title(f"splitter binary\ncount={row['expected_slice_count']}")
    for sl in split.slices:
        axes[1].add_patch(Rectangle((sl.x, sl.y), sl.width, sl.height, fill=False, edgecolor="cyan", linewidth=1.2))

    axes[2].imshow(entry["gray_view"], cmap="gray")
    axes[2].set_title("gray_x3_lanczos\n" + row["gray_text"])

    axes[3].imshow(entry["adaptive_view"], cmap="gray")
    axes[3].set_title("adaptive_threshold\n" + row["adaptive_text"])

    axes[4].imshow(entry["clahe_view"], cmap="gray")
    axes[4].set_title("clahe\n" + row["clahe_text"])

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("vertical_slice_text:", repr(row["vertical_slice_text"]))
    print("slice_boxes:", row["slice_boxes"])
    print("gap_count:", row["gap_count"], "split_confidence:", row["split_confidence"])
    print("vertical mean/min conf:", row["vertical_mean_conf"], row["vertical_min_conf"])


for debug_row in [1, 2, 3]:
    show_line_debug(debug_row)
